# Demo — O índice compartilhado: a trilha de indexação

**Objetivo:** percorrer a trilha de **indexação** do slide 9 (Ingestão → Extração → Chunking → Embedding → Indexação) e deixar pronto o índice `cap05-acervo`, que os notebooks 01 a 05 de `tipos-de-rag` consultam. Também ilustra os slides 10, 11, 12 e 23.

1. **Ingestão e chunking** → um trecho por seção de cada documento;
2. **Embedding** → cada trecho vira um vetor, e o modelo e a dimensão viram contrato;
3. **Definição do índice** → campos, HNSW, vetorizador e configuração semântica;
4. **Carga** → trechos e vetores entram no índice, sem duplicar quando o notebook roda de novo;
5. **Teste rápido** → uma consulta de texto prova que o índice responde.

> **Cenário 100% fictício.** A Contoso Industrial e todos os documentos, projetos, pessoas e números do `acervo/` existem apenas para fins didáticos.

A **extração** do slide 9 não aparece aqui porque o acervo já está em Markdown. Num acervo de PDFs, digitalizações e tabelas, ela seria feita pelo Document Intelligence, com o modelo Layout.

## 0. Preparação

Este notebook usa a **Azure OpenAI v1 API** (cliente Python `OpenAI`) para gerar os embeddings e o SDK `azure-search-documents` 12 para criar e carregar o índice.

Variáveis necessárias:

- `AZURE_OPENAI_ENDPOINT`, no formato `https://<recurso>.openai.azure.com`
- `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_DEPLOYMENT` (deployment de chat; não é usado aqui, mas a célula de configuração é a mesma em todos os notebooks)
- `AZURE_OPENAI_EMBEDDING_DEPLOYMENT`, `AZURE_OPENAI_EMBEDDING_MODEL` e `AZURE_OPENAI_EMBEDDING_DIMENSIONS`
- `AZURE_SEARCH_ENDPOINT` e `AZURE_SEARCH_API_KEY` (chave de **administração**: este notebook cria o índice)

Os valores vêm do arquivo `Cap05/.env` (copie o `.env.example` e preencha); o que faltar é perguntado na hora da execução.

**Tier do AI Search:** prefira o **Basic ou superior**. O semantic ranker (notebook 03) e o knowledge base (notebook 04) têm um plano gratuito, com cota mensal, em qualquer tier; no Free, só nas regiões que os oferecem nesse tier, como a Brazil South. O Basic dá mais índices e mais espaço, e só a partir dele existe o plano Standard, que libera o uso depois da cota.

> Não grave chaves diretamente no notebook.

In [ ]:
%pip install -q "openai>=2" "python-dotenv>=1.0" "pandas>=2.1" "tiktoken>=0.7" "azure-search-documents>=12.0.0,<13"

In [ ]:
import os
from getpass import getpass
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))  # lê o .env da pasta Cap05, se existir

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT") or input(
    "AZURE_OPENAI_ENDPOINT (ex.: https://meu-recurso.openai.azure.com): "
).strip()
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY") or getpass("AZURE_OPENAI_API_KEY: ")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT") or input(
    "AZURE_OPENAI_DEPLOYMENT (deployment de chat): "
).strip()
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT") or input(
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT (deployment de embedding): "
).strip()

# O modelo e a dimensão do embedding são o contrato entre indexação e consulta.
EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
EMBEDDING_DIMENSIONS = int(os.getenv("AZURE_OPENAI_EMBEDDING_DIMENSIONS", "1536"))

client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    base_url=f"{AZURE_OPENAI_ENDPOINT.rstrip('/')}/openai/v1/",
)

# Pasta Cap05: a primeira pasta, subindo a partir daqui, que contém o acervo.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "acervo").is_dir())
PASTA_ACERVO = RAIZ / "acervo"

print("Cliente configurado.")
print("Deployment de chat:", AZURE_OPENAI_DEPLOYMENT)
print("Embedding:", AZURE_OPENAI_EMBEDDING_DEPLOYMENT, f"({EMBEDDING_MODEL}, {EMBEDDING_DIMENSIONS} dimensões)")
print("Acervo:", PASTA_ACERVO)

In [ ]:
# Dois clientes: um administra o índice (criar, apagar), o outro grava e consulta documentos.
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient

AZURE_SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT") or input(
    "AZURE_SEARCH_ENDPOINT (ex.: https://meu-servico.search.windows.net): "
).strip()
AZURE_SEARCH_API_KEY = os.getenv("AZURE_SEARCH_API_KEY") or getpass("AZURE_SEARCH_API_KEY (chave de administração): ")

NOME_INDICE = "cap05-acervo"
credencial_busca = AzureKeyCredential(AZURE_SEARCH_API_KEY)
index_client = SearchIndexClient(AZURE_SEARCH_ENDPOINT, credencial_busca)
search_client = SearchClient(AZURE_SEARCH_ENDPOINT, NOME_INDICE, credencial_busca)

print("AI Search:", AZURE_SEARCH_ENDPOINT, "| índice:", NOME_INDICE)

---

# Execução 1 — Ingestão e chunking

A ingestão aqui é só ler os arquivos `.md` da pasta `acervo/`. A decisão que importa é o **chunking**: qual pedaço do documento vira uma unidade recuperável.

O acervo é feito de políticas, normas e procedimentos, todos já divididos em seções. É o caso em que o slide 12 indica o **chunking por estrutura do documento**: cada seção `## ` vira um trecho, com o título do documento, o nome da seção e o arquivo de origem.

### Fala sugerida

> “Chunking é a decisão mais barata de tomar e a mais cara de mudar.”

Trocar a regra de corte depois obriga a reprocessar e reindexar o acervo inteiro (slide 11).

In [ ]:
import re


def carregar_trechos(pasta_acervo: Path) -> list[dict]:
    """Lê os .md do acervo e devolve um trecho por seção "## " (chunking pela estrutura do documento)."""
    trechos = []
    for caminho in sorted(pasta_acervo.rglob("*.md")):
        fonte = caminho.relative_to(pasta_acervo).as_posix()
        doc_id = fonte.removesuffix(".md").replace("/", "_")
        texto = caminho.read_text(encoding="utf-8")
        titulo = texto.splitlines()[0].lstrip("# ").strip()
        # O que vem antes do primeiro "## " é o cabeçalho do documento (título e metadados).
        for numero, secao in enumerate(re.split(r"(?m)^## ", texto)[1:], start=1):
            nome_secao, _, conteudo = secao.partition("\n")
            trechos.append({
                "id": f"{doc_id}_{numero:02d}",
                "doc_id": doc_id,
                "titulo": titulo,
                "secao": nome_secao.strip(),
                "conteudo": conteudo.strip(),
                "fonte": fonte,
            })
    return trechos


trechos = carregar_trechos(PASTA_ACERVO)
print(f"{len(trechos)} trechos de {len({t['doc_id'] for t in trechos})} documentos")

In [ ]:
import tiktoken

pd.set_option("display.max_colwidth", None)  # tabelas mostram o texto inteiro

# Régua de tamanho: o tokenizador o200k_base, dos modelos GPT mais recentes.
codificador = tiktoken.get_encoding("o200k_base")
df_trechos = pd.DataFrame(trechos)
df_trechos["tokens"] = [len(codificador.encode(t["conteudo"])) for t in trechos]

display(df_trechos.groupby("fonte").agg(
    trechos=("id", "count"),
    tokens_min=("tokens", "min"),
    tokens_max=("tokens", "max"),
))

tokens = df_trechos["tokens"]
print(f"Tokens por trecho: mínimo {tokens.min()}, mediana {tokens.median():.0f}, máximo {tokens.max()}")

print("\nUm trecho, campo a campo:")
exemplo = next(t for t in trechos if t["id"] == "financeiro_reembolso-de-despesas-de-viagem_01")
for campo, valor in exemplo.items():
    print(f"{campo:>9}: {valor}")

### O que observar

- Cada documento vira poucos trechos, e cada trecho tem de ~30 a ~250 tokens de conteúdo. São pequenos porque o acervo é pequeno e bem dividido em seções.
- O ponto de partida que a Microsoft recomenda para o AI Search é **512 tokens com 25% de sobreposição** (slide 11). Aqui não há sobreposição: cada seção já é uma unidade de sentido completa.
- Os maiores trechos são da norma de TI, que tem subseções `###` dentro de cada `## `. Escolher em que nível de título cortar **é** a decisão de chunking.
- Cada trecho carrega `titulo`, `secao` e `fonte`: é isso que vai permitir citar a origem da resposta.

---

# Execução 2 — Embedding

Cada trecho vira um vetor de `EMBEDDING_DIMENSIONS` números, gerado pelo deployment de embedding (`text-embedding-3-small` no `.env.example`). O texto vetorizado junta o título do documento, a seção e o conteúdo, para que o vetor carregue também o contexto de onde o trecho veio.

In [ ]:
def texto_para_embedding(trecho: dict) -> str:
    """O texto que vira vetor: título do documento, seção e conteúdo."""
    return f"{trecho['titulo']} — {trecho['secao']}\n{trecho['conteudo']}"


def gerar_embeddings(textos: list[str], lote: int = 16) -> list[list[float]]:
    vetores = []
    for i in range(0, len(textos), lote):
        resposta = client.embeddings.create(
            model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
            input=textos[i:i + lote],
            dimensions=EMBEDDING_DIMENSIONS,
        )
        vetores.extend(item.embedding for item in resposta.data)
    return vetores


vetores = gerar_embeddings([texto_para_embedding(t) for t in trechos])
print(f"{len(vetores)} vetores, cada um com {len(vetores[0])} dimensões")
print("Os cinco primeiros números do primeiro vetor:", [round(x, 4) for x in vetores[0][:5]])

### Leitura arquitetural

- O modelo de embedding e a dimensão são o **contrato com a trilha de consulta**: cada pergunta precisa virar vetor com o mesmo modelo e a mesma dimensão, senão a comparação não tem sentido.
- Por isso todos os notebooks que geram embeddings leem as mesmas constantes `EMBEDDING_MODEL` e `EMBEDDING_DIMENSIONS`, do mesmo `.env`.
- **Trocar o modelo de embedding obriga a reindexar tudo**: vetores de modelos diferentes não se comparam (slide 10).

---

# Execução 3 — Definição do índice

O índice é onde essas decisões viram contrato:

- **Texto pesquisável** (`titulo`, `secao`, `conteudo`), com o analisador `pt-BR.lucene`: a busca por palavra-chave (BM25) do notebook 02.
- **Filtros** (`id`, `doc_id`, `fonte`): recortes por documento ou por origem.
- **Campo vetorial** `embedding`, com a dimensão do contrato, indexado com **HNSW** e cosseno: um grafo em memória que encontra vizinhos **aproximados**. Busca vetorial é aproximada por definição (slide 23).
- **Vetorizador** `vetorizador-aoai`: deixa o próprio AI Search transformar texto em vetor na hora da consulta. Os notebooks 01 a 03 vetorizam no cliente; quem precisa dele é o knowledge base do notebook 04, para vetorizar as consultas que recebe. O `resource_url` precisa estar no formato `https://<recurso>.openai.azure.com`, sem `/openai/v1` no final.
- **Configuração semântica** `semantica-padrao`: habilita o reranker (semantic ranker) do notebook 03.

> O vetorizador guarda a chave do Azure OpenAI dentro da definição do índice. Em produção, prefira identidade gerenciada.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    HnswAlgorithmConfiguration,
    HnswParameters,
    LexicalAnalyzerName,
    SearchableField,
    SearchField,
    SearchFieldDataType,
    SearchIndex,
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
    SemanticSearch,
    SimpleField,
    VectorSearch,
    VectorSearchAlgorithmMetric,
    VectorSearchProfile,
)

campos = [
    SimpleField(name="id", type=SearchFieldDataType.STRING, key=True, filterable=True),
    SimpleField(name="doc_id", type=SearchFieldDataType.STRING, filterable=True, facetable=True),
    SearchableField(name="titulo", analyzer_name=LexicalAnalyzerName.PT_BR_LUCENE),
    SearchableField(name="secao", analyzer_name=LexicalAnalyzerName.PT_BR_LUCENE),
    SearchableField(name="conteudo", analyzer_name=LexicalAnalyzerName.PT_BR_LUCENE),
    SimpleField(name="fonte", type=SearchFieldDataType.STRING, filterable=True),
    # O campo vetorial precisa ser SearchField: SimpleField e SearchableField ignoram a dimensão sem avisar.
    SearchField(
        name="embedding",
        type=SearchFieldDataType.Collection(SearchFieldDataType.SINGLE),
        searchable=True,
        vector_search_dimensions=EMBEDDING_DIMENSIONS,
        vector_search_profile_name="perfil-vetorial",
    ),
]

busca_vetorial = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(name="hnsw", parameters=HnswParameters(metric=VectorSearchAlgorithmMetric.COSINE)),
    ],
    profiles=[
        VectorSearchProfile(name="perfil-vetorial", algorithm_configuration_name="hnsw", vectorizer_name="vetorizador-aoai"),
    ],
    vectorizers=[
        AzureOpenAIVectorizer(
            vectorizer_name="vetorizador-aoai",
            parameters=AzureOpenAIVectorizerParameters(
                resource_url=AZURE_OPENAI_ENDPOINT.rstrip("/"),
                deployment_name=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
                api_key=AZURE_OPENAI_API_KEY,
                model_name=EMBEDDING_MODEL,
            ),
        ),
    ],
)

busca_semantica = SemanticSearch(
    default_configuration_name="semantica-padrao",
    configurations=[
        SemanticConfiguration(
            name="semantica-padrao",
            prioritized_fields=SemanticPrioritizedFields(
                title_field=SemanticField(field_name="titulo"),
                content_fields=[SemanticField(field_name="secao"), SemanticField(field_name="conteudo")],
            ),
        ),
    ],
)

indice = SearchIndex(name=NOME_INDICE, fields=campos, vector_search=busca_vetorial, semantic_search=busca_semantica)
indice_criado = index_client.create_or_update_index(indice)

print(f"Índice '{indice_criado.name}' criado ou atualizado. Campos, como o serviço os registrou:")
display(pd.DataFrame([
    {
        "campo": c["name"],
        "tipo": c["type"],
        "chave": bool(c.get("key")),
        "pesquisável": bool(c.get("searchable")),
        "filtrável": bool(c.get("filterable")),
        "facetável": bool(c.get("facetable")),
        "analisador": c.get("analyzer") or "",
        "dimensões": c.get("dimensions") or "",
    }
    for c in indice_criado.as_dict()["fields"]
]))

---

# Execução 4 — Carga dos trechos

Cada trecho entra no índice com o seu vetor. Usamos `merge_or_upload_documents`: como o `id` do trecho é estável (documento + número da seção), rodar o notebook de novo **atualiza** os mesmos documentos, em vez de duplicar.

A contagem do índice leva alguns segundos para refletir a carga; por isso esperamos até ela chegar ao número de trechos.

In [ ]:
import time

documentos = [{**trecho, "embedding": vetor} for trecho, vetor in zip(trechos, vetores)]
resultado_carga = search_client.merge_or_upload_documents(documents=documentos)

falhas = [r.key for r in resultado_carga if not r.succeeded]
print(f"Enviados: {len(resultado_carga)} | com sucesso: {len(resultado_carga) - len(falhas)} | falhas: {len(falhas)}")
if falhas:
    print("Trechos com falha:", falhas)

# A contagem é atualizada com atraso: esperamos até ~15 segundos.
for _ in range(15):
    total_no_indice = search_client.get_document_count()
    if total_no_indice >= len(trechos):
        break
    time.sleep(1)

print(f"Documentos no índice: {total_no_indice} (trechos no acervo: {len(trechos)})")
if total_no_indice > len(trechos):
    print("Há documentos a mais: sobras de uma carga anterior. merge_or_upload_documents não apaga nada.")

---

# Execução 5 — Teste rápido

Uma consulta de **texto** (BM25, sem vetor) com a palavra "reembolso", só para provar que o índice responde. Devem aparecer seções da política de viagens e reembolso. As consultas vetoriais começam no notebook 01.

In [ ]:
teste = search_client.search(search_text="reembolso", top=3, select=["id", "titulo", "secao"])

display(Markdown("**Consulta de texto:** `reembolso`"))
display(pd.DataFrame([
    {"score": round(r["@search.score"], 3), "id": r["id"], "título": r["titulo"], "seção": r["secao"]}
    for r in teste
]))

---

# Quadro final — o que foi decidido aqui, e quanto custa mudar depois

| Execução | O que vimos | Decisão arquitetural | Custo de mudar depois (slide 10) |
|---|---|---|---|
| 1 | Um trecho por seção `## `, de ~30 a ~250 tokens | Chunking por estrutura do documento | Reprocessar e reindexar tudo |
| 2 | Cada trecho vira um vetor com o modelo e a dimensão do `.env` | Modelo de embedding | Reindexar tudo: vetores de modelos diferentes não se comparam |
| 3 | Texto pesquisável, filtros, HNSW, vetorizador e configuração semântica | Base e índice | Migração, mas o conteúdo se aproveita |
| 4 | A carga roda de novo sem duplicar, porque o `id` do trecho é estável | Chave do trecho: documento + seção | Mudou a regra do `id`? Apagar e recarregar o índice |

## Frase de fechamento

> **O índice é o contrato entre as duas trilhas: mesmo modelo de embedding, mesmos campos, mesmos filtros**

---

# Referências oficiais

- Azure AI Search — criar um índice vetorial:  
  https://learn.microsoft.com/azure/search/vector-search-how-to-create-index

- Azure AI Search — chunking de documentos (512 tokens, 25% de sobreposição):  
  https://learn.microsoft.com/azure/search/vector-search-how-to-chunk-documents

- Azure AI Search — vetorizador do Azure OpenAI:  
  https://learn.microsoft.com/azure/search/vector-search-vectorizer-azure-open-ai

- Azure AI Search — configurar o semantic ranker:  
  https://learn.microsoft.com/azure/search/semantic-how-to-configure

---

# Limpeza (opcional)

**Os notebooks 01 a 05 dependem deste índice:** apague só no fim da aula. Se rodou o notebook 04, rode antes a limpeza dele, porque o knowledge base e o knowledge source apontam para este índice.

Mudou a dimensão do embedding no `.env`? Reinicie o kernel, porque os notebooks leem o `.env` só uma vez. Com a dimensão nova, o `create_or_update_index` falha: a dimensão de um campo vetorial não muda depois de criada. Apague o índice aqui e rode o notebook inteiro de novo, com `APAGAR_RECURSOS` de volta em `False`: é o "reindexar tudo" do slide 10.

Mudou só o modelo, com a mesma dimensão? Nada falha: o campo vetorial é o mesmo, e o vetorizador muda sem recriar o índice. Reinicie o kernel e rode o notebook inteiro, para regravar os 50 vetores com o modelo novo (slide 10: vetores de modelos diferentes não se comparam).

In [ ]:
APAGAR_RECURSOS = False  # os notebooks 01 a 05 dependem deste índice

if APAGAR_RECURSOS:
    index_client.delete_index(NOME_INDICE)
    print(f"Índice '{NOME_INDICE}' apagado.")
else:
    print(f"Nada foi apagado. O índice '{NOME_INDICE}' continua disponível para os notebooks 01 a 05.")